# Earthquake-cycle cluster pipeline — run & inspect

Thin monitoring notebook: it imports the same `pipeline` modules the CLIs use (single
source of truth) and lets you run a stage on a cluster, inspect picks/locations, and
render the regression comparison against the frozen baseline.

- Full chain from a shell: `python -m pipeline.cli.run_pipeline --cluster gwangyang`
- Regression only:        `python -m pipeline.cli.compare --cluster gwangyang`

Clusters: `gwangyang` (anchor), `kimcheon`, `jangsung`, `gyeongju`.

In [ ]:
import sys
sys.path.insert(0, "/home/msseo/works/10.Earthquake_cycle_project")
import pandas as pd
from pipeline import config, viz
from pipeline.core import pipeline
from pipeline.regression import compare

CLUSTER = "gwangyang"          # gwangyang | kimcheon | jangsung | gyeongju
cfg = config.load_cluster(CLUSTER)
print(cfg.name, "->", cfg.output_root)
print("velocity models:", [m.name for m in cfg.velocity_models])

## 1. (optional) run the chain
Skip if a run already exists under `pipeline/runs/<cluster>/`. PhaseNet runs on CPU
(matches the baseline). `--stage-from`/`--through` also available as `stage_from=`/`through=`.

In [ ]:
# pipeline.run_cluster(cfg, stage_from="stations", through="dtct", device="cpu")
# (uncomment to (re)run; a small slice: events=["20210720161418"])

## 2. Regression dashboard (new vs frozen baseline)
PASS/FAIL per stage. See `pipeline/README.md` for the determinism map and tolerances.

In [ ]:
df = compare.compare_all(cfg)
with pd.option_context("display.width", 200, "display.max_columns", 60):
    display(df)
print(f"{int(df.passed.sum())}/{len(df)} stages PASS")

## 3. Picks on a waveform
Defaults to the busiest station of the event; red=P, blue=S.

In [ ]:
import glob, os
event = sorted(os.path.basename(p).split("_")[0] for p in glob.glob(config.picks_dir(cfg)+"/*_picks.csv"))[0]
viz.plot_picks(cfg, event);

## 4. Catalog map + depth sections (HYPOINVERSE .sum)

In [ ]:
viz.map_catalog(cfg, velmodel="kim2011");
viz.depth_sections(cfg, velmodel="kim2011");
viz.cumulative_events(cfg, velmodel="kim2011");

## 5. Compare HYPOINVERSE velocity models
The framework runs N velocity models (Gwangyang: kim1983, kim2011 + 9 Seo2022).

In [ ]:
from pipeline.core import sumio
for vm in [m.name for m in cfg.velocity_models][:4]:
    try:
        s = sumio.read_sum(config.sum_file(cfg, vm))
        print(f"{vm:10s}: {len(s):3d} events | mean depth {s.depth.mean():.2f} km | mean RMS {s.rms.mean():.3f} s")
    except FileNotFoundError:
        print(f"{vm:10s}: not run")